In [ ]:
import ee
import geemap
from src.authorization.auth import authenticate_google_api, initialize_earth_engine
import traceback
from datetime import datetime, timezone
from dataclasses import dataclass
from pathlib import Path


@dataclass
class S2Config:
    datadir: Path
    collection: str
    scale: int
    cloud_perc: int
    bands: list[str]


class S2Downloader:
    def __init__(self, cfg: S2Config):
        self.cfg = cfg

    @staticmethod
    def build_roi(lon: float, lat: float, buffer_m: int) -> ee.Geometry:
        """
        This method creates circular buffer ROI of point that EE converts to bounding box.
        """
        return ee.Geometry.Point([lon, lat]).buffer(buffer_m).bounds()

    def find_image_id(self, roi: ee.Geometry, start_date: str, end_date: str) -> str | None:
        """
        Find image within collection and retrieve scene ID to download it.
        """
        col_img = (
            ee.ImageCollection(self.cfg.collection)
            .filterBounds(roi)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', self.cfg.cloud_perc))
            .sort('CLOUDY_PIXEL_PERCENTAGE')
            .first()
        )
        img_system_id = col_img.get('system:id').getInfo()

        return img_system_id

    @staticmethod
    def get_metadata_from_col_id(image_system_id: str) -> dict[str, str]:
        """
        Get metadata from image system id
        :param image_system_id: string
        :return: dictionary
        """
        img = ee.Image(image_system_id)
        metadata = img.toDictionary().getInfo()

        acquired_at = (
            datetime.fromtimestamp(
                metadata.get('GENERATION_TIME')/1000, tz=timezone.utc)
            .isoformat()
        )

        return {
            "system_id": image_system_id,
            "acquired_at": acquired_at,
            "cloud_pct": metadata.get("CLOUDY_PIXEL_PERCENTAGE"),
            "mgrs_tile": metadata.get("MGRS_TILE"),
            "product_id": metadata.get("PRODUCT_ID"),
            "platform": metadata.get("SPACECRAFT_NAME"),
            "processing_baseline": metadata.get("PROCESSING_BASELINE"),
        }

    def export_geotiff(self, image_id: str, image_roi: ee.Geometry) -> None:
        image_to_download = (ee.Image(image_id)
                             .select(['B4', 'B3', 'B2', 'B8'])
                             .clip(image_roi))
        output_path = self.cfg.datadir / f"{image_id}.tif"

        print(f'Downloading image... {image_to_download}')

        try:
            geemap.ee_export_image(
                image_to_download,
                filename=str(output_path),
                scale=10,
                region=image_roi,
                file_per_band=False)

            print('Image has been successfully downloaded to ', self.cfg.datadir)

        except Exception as e:
            traceback.print_exc()

In [ ]:
from src.tools.gee_utils import S2Config, S2Downloader
from pathlib import Path

#CURRENT_FILE = Path(__file__).resolve()
#PROJECT_ROOT = CURRENT_FILE.parents[1]
#DATA_DIR = PROJECT_ROOT / "data"
#DATA_DIR.mkdir(parents=True, exist_ok=True)

s2_config = S2Config(datadir='data',
                     collection='COPERNICUS/S2_SR_HARMONIZED',
                     scale=10,
                     cloud_perc=30,
                     bands=['B4', 'B3', 'B2', 'B8'],
                     roi_coordinates=[22.229681, 50.554120])

In [ ]:
type(s2_config.roi_coordinates)

In [ ]:
credentials = authenticate_google_api()
initialize_earth_engine(credentials)


In [ ]:
s2_downloader = S2Downloader(s2_config)


In [ ]:
roi = s2_downloader.build_roi(buffer_m=350)
img_system_id = s2_downloader.find_image_id(roi=roi,
                                            start_date='2022-04-01',
                                            end_date='2022-05-30')
scene_metadata = s2_downloader.get_metadata_from_col_id(img_system_id)
img_id = scene_metadata.get('product_id')

In [ ]:
print(scene_metadata)